In [ ]:
# Uncomment if running on drive

# !pip install -q "trl>=0.15.0" "transformers>=5.0.0" accelerate datasets peft qwen-vl-utils

# from google.colab import drive
# drive.mount('/content/drive', force_remount=True)
# %cd /content/drive/MyDrive/RL/RL_Scene_Graphs
# %ls


# import sys
# from pathlib import Path
# import os
# os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

# PROJECT_DIR = Path("/content/drive/MyDrive/RL/RL_Scene_Graphs")  # change this
# sys.path.append(str(PROJECT_DIR))

import torch
import torch.nn.functional as F
from datasets import load_from_disk, Dataset
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration, Qwen2VLForConditionalGeneration, TrainingArguments, Trainer
from trl import GRPOTrainer, GRPOConfig
from peft import LoraConfig, TaskType
from tqdm import tqdm
import json

from mllm_generation import (
    build_messages,
    build_inputs,
    extract_answer_content
)
from mllm_eval import total_reward
import ast


Mounted at /content/drive
/content/drive/MyDrive/RL/RL_Scene_Graphs
datasets/      Image_Data_Prep.ipynb  MLLM_Pipeline.ipynb
grpo-run/      MLLM_Eval.ipynb        MLLM_Train_GRPO.ipynb
grpo-run-sft/  mllm_eval.py           MLLM_Train_GRPO_SFT.ipynb
grpo-sgg/      mllm_generation.py     __pycache__/


### Dataset and Prompt

### Load Dataset

In [ ]:

iterable_ds = load_from_disk("./datasets/vg150_val_sgg_prompt").to_iterable_dataset()

train_ds = Dataset.from_list(list(iterable_ds.take(1500)))

print(train_ds)
# print(test_ds)

print("Training examples: ", len(train_ds))
print(train_ds[0])
# print("Test examples: ", len(test_ds))
# print(test_ds[0])
# print an example


Dataset({
    features: ['image_id', 'image', 'prompt_open', 'prompt_close', 'objects', 'relationships'],
    num_rows: 1500
})
Training examples:  1500
{'image_id': '1', 'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=800x600 at 0x7D0366E83E30>, 'prompt_open': 'Generate a structured scene graph for an image of size (800 x 600) using the following format:\n\n<answer>\n{\n  "objects": [\n    {"id": "object_name.number", "bbox": [x1, y1, x2, y2]},\n    ...\n  ],\n  "relationships": [\n    {"subject": "object_name.number", "predicate": "relationship_type", "object": "object_name.number"},\n    ...\n  ]\n}\n</answer>\n\n### **Guidelines:**\n- **Objects:**\n  - Assign a unique ID for each object using the format `"object_name.number"` (e.g., `"person.1"`, `"bike.2"`).\n  - Provide its bounding box `[x1, y1, x2, y2]` in integer pixel format.\n  - Include all visible objects, even if they have no relationships.\n\n- **Relationships:**\n  - Represent interactions accurately usi

In [ ]:
PROMPT_TEMPLATE = """Generate a scene graph for the image of size ({width} x {height}).

Output ONLY:

<answer>
{{
  "objects": [{{"id": "name.number", "bbox": [x1, y1, x2, y2]}}],
  "relationships": [{{"subject": "name.number", "predicate": "relation", "object": "name.number"}}]
}}
</answer>

Rules:
- Unique ids: name.number
- Bounding boxes must be integers
- Include all visible objects
- Relationships must reference valid object ids
- Output valid JSON only
"""

def build_dataset(raw_ds):
    data = []

    for sample in tqdm(raw_ds, total=len(raw_ds)):
        width, height = sample["image"].size
        prompt = PROMPT_TEMPLATE.format(width=width, height=height)
        data.append({
            "prompt": sample["prompt_open"],
            # "prompt": prompt,
            "image_data": sample["image"],   # rename
            "solution": {
                "objects": ast.literal_eval(sample["objects"]),
                "relationships": ast.literal_eval(sample["relationships"])
            }
        })

    return Dataset.from_list(data)

train_ds = build_dataset(train_ds)
# test_ds = build_dataset(test_ds)

print(train_ds)
# print(test_ds)



100%|██████████| 1500/1500 [00:05<00:00, 280.75it/s]


Dataset({
    features: ['prompt', 'image_data', 'solution'],
    num_rows: 1500
})


In [ ]:
SYSTEM_PROMPT = (
    "You are a vision-language assistant that generates structured scene graphs from images. "
    "Always output valid JSON with 'objects' and 'relationships'."
)

### Reward

In [ ]:
def grpo_reward(completions, solution, **kwargs):
    rewards = []

    for i, (completion, gt) in enumerate(zip(completions, solution)):
        if isinstance(completion, str):
            text = completion
        else:
            text = completion[0]["content"]

        try:
            parsed = extract_answer_content(text)
            pred = json.loads(parsed)
        except:
            pred = {"_raw_text": text}

        print(f"\n--- SAMPLE {i} ---")
        print("TEXT:", text)
        print("Reward with GT: ", gt, type(gt), " and pred ", pred, type(pred))

        r = total_reward(pred, gt)
        r_tot = r['total']


        print("REWARD:", r)

        rewards.append(r_tot)

    return rewards

### Custom Trainer

In [ ]:
class GRPOConfigVL(TrainingArguments):
    def __init__(
        self,

        # ===== CORE GRPO =====
        num_generations=4,                 # 🔥 increase from 2 → 4
        max_completion_length=256,         # 🔥 reduce for memory + stability
        beta=0.02,                        # KL penalty (small but non-zero)
        epsilon=0.2,                      # PPO-style clipping

        # ===== GENERATION =====
        temperature=0.9,                  # 🔥 slightly lower = more stable
        top_p=1.0,
        top_k=50,
        repetition_penalty=1.0,
        do_sample=True,

        # ===== REWARD =====
        scale_rewards=True,               # 🔥 important
        normalize_advantages=True,        # custom flag for your code

        # ===== TRAINING STABILITY =====
        loss_scale=1.0,                   # optional scaling
        clip_logprobs_min=-50.0,          # 🔥 avoid extreme values

        # ===== DEBUG =====
        log_completions=True,

        **kwargs
    ):
        super().__init__(**kwargs)

        self.num_generations = num_generations
        self.max_completion_length = max_completion_length

        self.beta = beta
        self.epsilon = epsilon

        self.temperature = temperature
        self.top_p = top_p

        self.scale_rewards = scale_rewards
        self.normalize_advantages = normalize_advantages

        self.loss_scale = loss_scale
        self.clip_logprobs_min = clip_logprobs_min

        self.log_completions = log_completions

In [ ]:
class GRPOTrainerVL(Trainer):

  def __init__(
        self,
        model,
        args,
        train_dataset=None,
        eval_dataset=None,
        processing_class=None,
        reward_funcs=None,
        **kwargs,
    ):
        super().__init__(
            model=model,
            args=args,
            train_dataset=train_dataset,
            eval_dataset=eval_dataset,
            **kwargs,
        )

        # 🔥 custom components
        self.processing_class = processing_class
        self.reward_funcs = reward_funcs

  def generate_completions(self, batch):
    prompts = batch["prompt"]
    images = batch["image_data"]

    all_prompt_ids = []
    all_completion_ids = []
    all_texts = []

    for prompt, image in zip(prompts, images):

        for ind in range(self.args.num_generations):

            print("GEN ", ind)

            messages = build_messages(image, prompt, system_prompt=SYSTEM_PROMPT)
            inputs, _ = build_inputs(self.processing_class, messages, self.model.device)

            self.model.eval()
            with torch.no_grad():
                outputs = self.model.generate(
                    **inputs,
                    max_new_tokens=self.args.max_completion_length,
                    do_sample=self.args.do_sample,
                    temperature=self.args.temperature,
                    top_p=self.args.top_p,
                    top_k=self.args.top_k,
                    repetition_penalty=self.args.repetition_penalty,
                )
            self.model.train()

            input_ids = inputs["input_ids"][0]
            gen_ids = outputs[0, input_ids.shape[0]:]

            text = self.processing_class.batch_decode(
                [gen_ids], skip_special_tokens=True
            )[0]

            all_prompt_ids.append(input_ids)
            all_completion_ids.append(gen_ids)
            all_texts.append(text)

    print("GENERATED --- ")
    print(len(all_prompt_ids), " = prompt ids ", type(all_prompt_ids[0]))
    print(len(all_completion_ids), " = completion ids ", type(all_completion_ids[0]))
    print(len(all_texts), " = texts")
    return all_prompt_ids, all_completion_ids, all_texts


  def compute_logprobs(self, prompt_ids, completion_ids):
    device = self.model.device

    # ===== Build full sequence =====
    full_ids = torch.cat([prompt_ids, completion_ids], dim=0).unsqueeze(0)
    attention_mask = torch.ones_like(full_ids)

    prompt_len = prompt_ids.shape[0]
    completion_len = completion_ids.shape[0]

    # ===== Build labels mask =====
    labels = full_ids.clone()

    # mask prompt tokens → ignored in loss
    labels[:, :prompt_len] = -100

    # ===== Forward pass (WITH grad) =====
    outputs = self.model(
        input_ids=full_ids,
        attention_mask=attention_mask,
        labels=labels,   # 🔥 KEY
    )

    # logits: (1, P+C, V)
    logits = outputs.logits
    # 🔥 temperature scaling (important)
    logits = logits / self.args.temperature
    log_probs = torch.log_softmax(logits, dim=-1)

    # ===== Extract ONLY completion token logprobs =====
    token_logprobs = []

    for t in range(completion_len):
        token_id = int(completion_ids[t])
        pos = prompt_len + t - 1
        lp = log_probs[0, pos, token_id]

        # 🔥 clamp logprobs (VERY important)
        lp = torch.clamp(lp, min=self.args.clip_logprobs_min)

        token_logprobs.append(lp)

    return torch.stack(token_logprobs)  # (C,)

  def compute_rewards(self, prompts, completions, batch):

    expanded_solutions = []

    for sol in batch["solution"]:      # length B
        for _ in range(self.args.num_generations):
            expanded_solutions.append(sol)

    rewards = self.reward_funcs(
        completions=[[{"content": c}] for c in completions],
        solution=expanded_solutions
    )
    return torch.tensor(rewards, device=self.model.device)

  def compute_loss(self, model, inputs, return_outputs=False, **kwargs):

    prompts = inputs["prompt"]

    prompt_ids_list, completion_ids_list, texts = self.generate_completions(inputs)

    # ===== rewards =====
    rewards = self.compute_rewards(prompts, texts, inputs)

    # ===== reshape into groups =====
    G = self.args.num_generations
    rewards = rewards.view(-1, G) # B,G

    advantages = rewards - rewards.mean(dim=1, keepdim=True)
    if self.args.normalize_advantages:
        advantages = advantages / (rewards.std(dim=1, keepdim=True) + 1e-8)

    advantages = advantages.view(-1) # (B*G,1)

    # ===== compute logprobs =====
    all_logprobs = []

    for p_ids, c_ids in zip(prompt_ids_list, completion_ids_list):
        lp = self.compute_logprobs(p_ids, c_ids)
        # 🔥 USE MEAN NOT SUM
        seq_logprob = lp.mean()

        all_logprobs.append(seq_logprob)  # sequence logprob

    logprobs = torch.stack(all_logprobs)

    # ===== PPO-style ratio =====
    old_logprobs = logprobs.detach()

    ratio = torch.exp(logprobs - old_logprobs)

    # ===== clipping =====
    epsilon = self.args.epsilon

    clipped_ratio = torch.clamp(ratio, 1 - epsilon, 1 + epsilon)

    loss_1 = ratio * advantages
    loss_2 = clipped_ratio * advantages

    loss = -torch.min(loss_1, loss_2).mean()

    # ===== debug =====
    print("Rewards:", rewards[:3])
    print("Advantages:", advantages[:6])
    print("Logprobs:", logprobs[:6])
    print("Loss:", loss.item())
    print("loss requires grad:", loss.requires_grad)

    return loss

### Model - Option 1 - Load pretrained SFT model from Paper

In [ ]:
model_name = "JosephZ/R1-SGG-Zero-7B"

processor = AutoProcessor.from_pretrained(model_name)

model = Qwen2VLForConditionalGeneration.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

print("DEVICE =", model.device)
model.train()   # for RL

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/730 [00:00<?, ?it/s]

DEVICE = cuda:0


Qwen2VLForConditionalGeneration(
  (model): Qwen2VLModel(
    (visual): Qwen2VisionTransformerPretrainedModel(
      (patch_embed): PatchEmbed(
        (proj): Conv3d(3, 1280, kernel_size=(2, 14, 14), stride=(2, 14, 14), bias=False)
      )
      (rotary_pos_emb): VisionRotaryEmbedding()
      (blocks): ModuleList(
        (0-31): 32 x Qwen2VLVisionBlock(
          (norm1): LayerNorm((1280,), eps=1e-06, elementwise_affine=True)
          (norm2): LayerNorm((1280,), eps=1e-06, elementwise_affine=True)
          (attn): VisionAttention(
            (qkv): Linear(in_features=1280, out_features=3840, bias=True)
            (proj): Linear(in_features=1280, out_features=1280, bias=True)
          )
          (mlp): VisionMlp(
            (fc1): Linear(in_features=1280, out_features=5120, bias=True)
            (act): QuickGELUActivation()
            (fc2): Linear(in_features=5120, out_features=1280, bias=True)
          )
        )
      )
      (merger): PatchMerger(
        (ln_q): LayerN

### Model - Option 2 - Use Base model (7B or 3B)

In [ ]:
# model_name = "Qwen/Qwen2.5-VL-3B-Instruct" # use smaller model
model_name = "Qwen/Qwen2.5-VL-7B-Instruct" # use larger model


processor = AutoProcessor.from_pretrained(model_name)


model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

print("DEVICE = ", model.device)
model.train()




### Lora (optional)

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.enable_input_require_grads()
model.print_trainable_parameters()

trainable params: 5,046,272 || all params: 8,296,421,888 || trainable%: 0.0608


### GRPO Config

In [ ]:

def grpo_data_collator(features):
    batch = {}

    batch["prompt"] = [f["prompt"] for f in features]
    batch["image_data"] = [f["image_data"] for f in features]
    batch["solution"] = [f["solution"] for f in features]

    return batch

config = GRPOConfigVL(
    output_dir="./grpo-run-sft",

    # ===== TRAINING =====
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,   # helps stability
    num_train_epochs=1,

    learning_rate=1e-6,

    # ===== GRPO CORE =====
    num_generations=4,               # 🔥 VERY IMPORTANT (was 2)
    max_completion_length=1024,       # 🔥 reduce from 1024

    beta=0.02,                      # (unused for now, future KL)
    epsilon=0.2,                    # PPO clipping

    # ===== GENERATION =====
    temperature=0.9,
    top_p=1.0,
    top_k=50,
    repetition_penalty=1.0,
    do_sample=True,

    # ===== STABILITY =====
    normalize_advantages=True,
    clip_logprobs_min=-50.0,

    # ===== LOGGING =====
    logging_steps=1,
    save_steps=50,

    remove_unused_columns=False,
)

### Training

In [ ]:
trainer = GRPOTrainerVL(
    model=model,
    args=config,
    train_dataset=train_ds,
    processing_class=processor,
    reward_funcs=grpo_reward,
    data_collator=grpo_data_collator,
)

trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 151645, 'bos_token_id': None, 'pad_token_id': 151643}.


GEN  0
GEN  1
GEN  2
GEN  3
GENERATED --- 
4  = prompt ids  <class 'torch.Tensor'>
4  = completion ids  <class 'torch.Tensor'>
4  = texts

--- SAMPLE 0 ---
TEXT: <answer>
{
  "objects": [
    {"id": "table.1", "bbox": [280, 433, 1003, 763]},
    {"id": "table.2", "bbox": [110, 373, 653, 593]},
    {"id": "table.3", "bbox": [543, 383, 843, 483]},
    {"id": "table.4", "bbox": [543, 693, 1003, 1003]},
    {"id": "chair.1", "bbox": [193, 733, 533, 1003]},
    {"id": "chair.2", "bbox": [573, 543, 803, 873]},
    {"id": "chair.3", "bbox": [783, 403, 853, 483]},
    {"id": "chair.4", "bbox": [803, 473, 1003, 723]},
    {"id": "whiteboard.1", "bbox": [343, 143, 603, 373]},
    {"id": "whiteboard.2", "bbox": [123, 73, 323, 323]},
    {"id": "board.1", "bbox": [0, 63, 513, 373]},
    {"id": "light", "bbox": [0, 0, 563, 73]}
  ],
  "relationships": [
    {"subject": "table.1", "predicate": "near", "object": "chair.1"},
    {"subject": "table.1", "predicate": "near", "object": "chair.2"},
    {"s

Step,Training Loss
1,0.000000


GEN  0
GEN  1
GEN  2
GEN  3
GENERATED --- 
4  = prompt ids  <class 'torch.Tensor'>
4  = completion ids  <class 'torch.Tensor'>
4  = texts

--- SAMPLE 0 ---
TEXT: <answer>
{
  "objects": [
    {"id": "table.1", "bbox": [0, 533, 703, 1003]},
    {"id": "chair.1", "bbox": [103, 683, 343, 1003]},
    {"id": "chair.2", "bbox": [263, 793, 473, 1003]},
    {"id": "plate", "bbox": [133, 583, 273, 663]},
    {"id": "tv", "bbox": [483, 413, 683, 673]},
    {"id": "cabinet", "bbox": [513, 553, 903, 893]},
    {"id": "wall", "bbox": [0, 0, 1003, 693]},
    {"id": "jar", "bbox": [563, 47, 623, 153]},
    {"id": "shelf", "bbox": [413, 123, 743, 193]},
    {"id": "lamp", "bbox": [523, 643, 563, 713]}
  ],
  "relationships": [
    {"subject": "table.1", "predicate": "has", "object": "plate"},
    {"subject": "table.1", "predicate": "has", "object": "chair.1"},
    {"subject": "table.1", "predicate": "has", "object": "chair.2"},
    {"subject": "chair.1", "predicate": "at", "object": "table.1"},
    {"

KeyboardInterrupt: 